In [1]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import SaltRemover
from rdkit.Chem.rdmolops import RemoveHs
from rdkit.Chem import rdmolops
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt
from collections import defaultdict

df=pd.read_csv('herg_row_data52.csv')

delited_data={
    'delited_mol':0,
    'delited_isotops': 0,
    'delited_inorganic':0,
    'delited_nonallowed_elements':0,
    'delited_Stereo':0,
    'delited_Radical':0
}
    
def standardized_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        if mol is None:
            delited_data['delited_mol'] += 1
            return None

        # Разделение на фрагменты и выбор наибольшего
        mol_frags = rdmolops.GetMolFrags(mol, asMols=True)
        largest_mol = max(mol_frags, default=mol, key=lambda m: m.GetNumAtoms())

        # Нейтрализация зарядов (только largest_mol)
        def neutralize_atoms(m):
            pattern = Chem.MolFromSmarts("[+1!h0!$([*]~[-1,-2,-3,-4]),-1!$([*]~[+1,+2,+3,+4])]")
            at_matches = m.GetSubstructMatches(pattern)
            for match in at_matches:
                atom = m.GetAtomWithIdx(match[0])
                chg = atom.GetFormalCharge()
                hcount = atom.GetTotalNumHs()
                atom.SetFormalCharge(0)
                atom.SetNumExplicitHs(max(0, hcount - chg))  # Защита от отрицательного числа Hs
            return m

        largest_mol = neutralize_atoms(largest_mol)

        # Удаление изотопных меток (только largest_mol)
        delited_isotops = False
        for atom in largest_mol.GetAtoms():
            if atom.GetIsotope() != 0:
                atom.SetIsotope(0)
                delited_isotops = True
        if delited_isotops:
            delited_data['delited_isotops'] += 1

        # Удаление водородов
        largest_mol = RemoveHs(largest_mol)

        # Проверка на органическую природу (есть ли углерод)
        if not any(atom.GetAtomicNum() == 6 for atom in largest_mol.GetAtoms()):
            delited_data['delited_inorganic'] += 1
            return None

        # Проверка элементов (только largest_mol)
        allowed_elements = {1, 5, 6, 7, 8, 9, 15, 16, 17, 35, 53, 11, 19}
        for atom in largest_mol.GetAtoms():
            if atom.GetAtomicNum() not in allowed_elements:
                delited_data['delited_nonallowed_elements'] += 1
                return None

        # Удаление радикалов
        def remove_radicals(m):
            for atom in m.GetAtoms():
                if atom.GetNumRadicalElectrons() > 0:
                    delited_data['delited_Radical'] += 1
                    return None
            return m

        largest_mol = remove_radicals(largest_mol)
        if largest_mol is None:
            return None

        # Удаление стереохимии
        if any(atom.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED for atom in largest_mol.GetAtoms()):
            Chem.RemoveStereochemistry(largest_mol)
            delited_data['delited_Stereo'] += 1

        # Финальная санетизация и генерация SMILES
        Chem.SanitizeMol(largest_mol, Chem.SANITIZE_ALL)
        standardized_smiles = Chem.MolToSmiles(largest_mol, isomericSmiles=False, canonical=True)

        return standardized_smiles
    except :
        return None

df['standardized_smiles'] = df['canonical_smiles'].apply(standardized_smiles)
df = df.dropna(subset=['standardized_smiles'])

for step, count in delited_data.items():
    if count > -1:
        print(f"{step}: {count}")


delited_mol: 0
delited_isotops: 14
delited_inorganic: 0
delited_nonallowed_elements: 7
delited_Stereo: 4692
delited_Radical: 0


In [2]:
duplicates_stats = {
    'total_smiles': 0,          # количество уникальных смайлс 
    'duplicate_smiles': 0,      # количество смайлс с дубликатами
    'duplicate_entries': 0,     # количество дублирующихся записей 
    'unique_pairs': [],         # уникальное значение смайл-логарифм 
}

data_without_duplicates = defaultdict(list)
for standardized_smiles, logBB_value in df[['standardized_smiles', 'pIC50']].values:
    data_without_duplicates[standardized_smiles].append(logBB_value)
for smiles, values in data_without_duplicates.items():
    duplicates_stats['total_smiles'] += 1
    
    if len(values) > 1:
        duplicates_stats['duplicate_smiles'] += 1
        duplicates_stats['duplicate_entries'] += len(values) - 1
    else:
        duplicates_stats['unique_pairs'].append((smiles, values[0]))

print(f"Всего уникальных smiles: {duplicates_stats['total_smiles']}")
print(f"smiles с дубликатами: {duplicates_stats['duplicate_smiles']}")
print(f"Всего дублирующихся записей: {duplicates_stats['duplicate_entries']}")
print(f"Уникальных пар smiles-pIC50: {len(duplicates_stats['unique_pairs'])}")

Всего уникальных smiles: 10126
smiles с дубликатами: 808
Всего дублирующихся записей: 1215
Уникальных пар smiles-pIC50: 9318


In [3]:
from itertools import chain
from numpy import asarray, mean
from sklearn.neighbors import BallTree
import pandas as pd

# преобразование в числовой формат, удаление пропусков
df['pIC50'] = pd.to_numeric(df['pIC50'], errors='coerce')
df_clean = df.dropna(subset=['pIC50'])

def mean_pIC50(y, threshold=0.5):
    try:
        unique_values = list(set([float(x) for x in y if pd.notna(x)]))

        if len(unique_values) == 1:
            return unique_values[0], None
        
        
        if not unique_values:
            return None, None
        yy = asarray(unique_values).reshape(-1, 1)
        if len(yy) < 1:
            return None, None
            
        tree = BallTree(yy, metric='manhattan')
        indexes, distances = tree.query_radius(yy, threshold, return_distance=True)
        unique_groups = set([tuple(x) for x in indexes])
        unique_groups_count = {i: (x, len(x)) for i, x in enumerate(unique_groups)}
        
        if len(yy) == 1:
            return yy[0][0], None
        elif len(unique_groups) > 2:
            return None, None
        elif len(unique_groups) == 1:
            return round(mean(unique_values), 2), None
        elif unique_groups_count[0][1] > unique_groups_count[1][1]:
            main_group = unique_groups_count[0][0]
            outliers = [x for x in chain(*indexes) if x not in main_group]
            return round(mean(yy[list(main_group)]), 2), outliers
        elif unique_groups_count[0][1] < unique_groups_count[1][1]:
            main_group = unique_groups_count[1][0]
            outliers = [x for x in chain(*indexes) if x not in main_group]
            return round(mean(yy[list(main_group)]), 2), outliers
        else:
            return None, None
    except Exception as e:
        print(f"Error processing molecule: {e}")
        return None


results = []
for smiles, group in df_clean.groupby('standardized_smiles'):
    if not group.empty:
       # если одно значение его берем как есть, а если несколько значение pIC50 то применяем функцию
        if len(group['pIC50']) == 1: 
            mean_val = float(group['pIC50'].iloc[0])
        else:
            # Для нескольких значений применяем функцию
            mean_val, _ = mean_pIC50(group['pIC50'], threshold=0.5)
        if mean_val is not None:
            results.append({'standardized_smiles': smiles, 'mean_pIC50': mean_val})

result_df = pd.DataFrame(results)




df_final = pd.merge(df, result_df, on='standardized_smiles',how='inner')

df_final = df_final.drop_duplicates(subset=['standardized_smiles'])
print(df_final['standardized_smiles'].duplicated().sum())
print(df_final.head(2))


0
  molecule_chembl_id                                   canonical_smiles  \
0       CHEMBL443476  O=C1NCCN1CCN1CCC(c2cn(-c3ccccc3)c3ccc(Cl)cc23)CC1   
1        CHEMBL53661   O=C1NCCN1CCN1CCC(c2cn(C3CCCCC3)c3ccc(Cl)cc23)CC1   

   standard_value standard_units assay_type     pIC50  \
0            88.0             nM          B  7.055517   
1           137.0             nM          B  6.863279   

                                 standardized_smiles  mean_pIC50  
0  O=C1NCCN1CCN1CCC(c2cn(-c3ccccc3)c3ccc(Cl)cc23)CC1        7.06  
1   O=C1NCCN1CCN1CCC(c2cn(C3CCCCC3)c3ccc(Cl)cc23)CC1        6.86  


In [4]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.io as pio

# Функция для расчета молекулярной массы
def mass_mol(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Descriptors.MolWt(mol) if mol else None
    except:
        return None

# Применяем функцию и фильтруем данные
df_final['molecular_weight'] = df_final['standardized_smiles'].apply(mass_mol)
print(f"Размер до фильтрации по массе: {df_final.shape}")


df_final = df_final[(df_final['molecular_weight'] >= 50) & (df_final['molecular_weight'] <= 850)].copy()
print(f"Размер после фильтрации по массе: {df_final.shape}")

# Создаем субплотовую структуру
fig = sp.make_subplots(rows=1, cols=2, subplot_titles=(
    'Распределение молекулярной массы', 
    'Распределение pIC50'
))

# Добавляем гистограммы
fig.add_trace(
    go.Histogram(
        x=df_final['molecular_weight'],
        nbinsx=50,
        marker_color='#1f77b4',
        name='Молекулярная масса'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=df_final['pIC50'],
        nbinsx=50,
        marker_color='#ff7f0e',
        name='pIC50'
    ),
    row=1, col=2
)

# Настраиваем оформление
fig.update_layout(
    showlegend=False,
    template='plotly_white',
    height=500,
    width=900
)

fig.update_xaxes(title_text='Молекулярная масса (г/моль)', row=1, col=1)
fig.update_yaxes(title_text='Количество молекул', row=1, col=1)
fig.update_xaxes(title_text='pIC50', row=1, col=2)
fig.update_yaxes(title_text='Количество молекул', row=1, col=2)

# Настройка отображения
pio.renderers.default = 'browser'  # Открывает в браузере
fig.show()

# Сохраняем результаты (исправлено df_final вместо df)
print(f"\nFinal dataset shape: {df_final.shape}")
print(df_final.head(2))


Размер до фильтрации по массе: (9915, 9)
Размер после фильтрации по массе: (9904, 9)

Final dataset shape: (9904, 9)
  molecule_chembl_id                                   canonical_smiles  \
0       CHEMBL443476  O=C1NCCN1CCN1CCC(c2cn(-c3ccccc3)c3ccc(Cl)cc23)CC1   
1        CHEMBL53661   O=C1NCCN1CCN1CCC(c2cn(C3CCCCC3)c3ccc(Cl)cc23)CC1   

   standard_value standard_units assay_type     pIC50  \
0            88.0             nM          B  7.055517   
1           137.0             nM          B  6.863279   

                                 standardized_smiles  mean_pIC50  \
0  O=C1NCCN1CCN1CCC(c2cn(-c3ccccc3)c3ccc(Cl)cc23)CC1        7.06   
1   O=C1NCCN1CCN1CCC(c2cn(C3CCCCC3)c3ccc(Cl)cc23)CC1        6.86   

   molecular_weight  
0           422.960  
1           429.008  


In [5]:
# меньше 5 значит идельно безопасный pIC50 , если pIC50  больше или равно 5, то опасный 
# Toxic=1
# Save=0
activity =[]
for a in df_final['pIC50']:
    if a >= 5:
        activity.append('1')
    else :
        activity.append('0')
df_final['activity']= activity 

print(df_final['activity'].value_counts())

df_final.to_csv('herg_num_data.csv', index=False)

activity
1    5588
0    4316
Name: count, dtype: int64
